# Slow spectral correction experiment — v3-beta review

Question: does a time-constant, more flexible low-frequency correction reduce the 50–70 MHz exclusions without fitting away interference or losing model support?

The baseline is the current package algorithm (50 ns frequency half-width, 1 mHz time half-width). The candidate adds a frequency DPSS basis restricted to 35–88 MHz, tapered over 5 MHz at either end, with **one constant time coefficient per added spectral mode**. All coefficients are fitted jointly. Extra spectral modes are orthogonalized against the original frequency basis. The support calculation uses the complete combined design. Point, cross, group, and guard thresholds are unchanged.

Candidate widths are 150 and 300 ns. These are diagnostic trials, not changes to package defaults. Checkpoints use offsets 1, 4 and 10 hours in visit 13, each for 20 minutes. Every 37th integration and a contiguous 20-second block are withheld in the prediction check. Acceptance requires improved held-out low-band residuals, useful retention, preserved support and preserved injected-event detection; retaining more data alone is not acceptance.

In [ ]:
from pathlib import Path
import sys, time, hashlib, inspect, json, subprocess, warnings
from dataclasses import replace, asdict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from scipy.linalg import eigh
from scipy.sparse.linalg import LinearOperator, cg
from scipy.ndimage import maximum_filter
from hera_filters.dspec import dpss_operator
from IPython.display import display
%matplotlib widget
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'eigsep_data/src').is_dir())
sys.path.insert(0,str(ROOT/'eigsep_data/src'))
from eigsep_data import MetadataIndex
from eigsep_data import rfi_supported as base
from threadpoolctl import threadpool_limits
thread_limit=threadpool_limits(limits=1)
P=base.RFIConfig()
CAMPAIGN=ROOT/'marjum-2026-07'
print('Source SHA256:',hashlib.sha256(Path(base.__file__).read_bytes()).hexdigest())
print('Package commit:',subprocess.check_output(['git','-C',str(ROOT/'eigsep_data'),'rev-parse','HEAD'],text=True).strip())
display(pd.Series(asdict(P)))


In [ ]:
class MixedFit(base._TensorFit):
    def __init__(self,times,freqs,data,config,width):
        super().__init__(times,freqs,data,config)
        self.cross_qt=self.qt.copy()
        nt,nf=self.shape
        j=(freqs>=35)&(freqs<88)
        extra=np.zeros((len(freqs),1))
        if width:
            op=dpss_operator(freqs[j]*1e6,[0],[width],eigenval_cutoff=[config.eigenvalue_cutoff])[0].real
            taper=np.minimum(np.clip((freqs[j]-35)/5,0,1),np.clip((88-freqs[j])/5,0,1))
            taper=np.sin(np.pi*taper/2)**2
            extra=np.zeros((len(freqs),op.shape[1]))
            extra[j]=op*taper[:,None]/self.scale[j,None]
            extra-=self.qf@(self.qf.T@extra)
            u,s,_=np.linalg.svd(extra,full_matrices=False)
            extra=u[:,s>s[0]*1e-8]
            self.qf=np.column_stack([self.qf,extra])
            self.qt=np.column_stack([self.qt,np.ones(len(times))/np.sqrt(len(times))])
        self.shape=(self.qt.shape[1],self.qf.shape[1])
        mask=np.zeros(self.shape,bool);mask[:nt,:nf]=True
        if width: mask[nt,nf:]=True
        self.active=np.flatnonzero(mask.ravel())
        self.active_mask=mask
        self.ncoeff=len(self.active)

    def expand(self,c):
        out=np.zeros(np.prod(self.shape));out[self.active]=c
        return out.reshape(self.shape)

    def solve(self,data,keep):
        start=time.perf_counter(); qt,qf=self.qt,self.qf;ridge=self.config.ridge
        if keep.sum()<=self.ncoeff: raise ValueError('Insufficient training data')
        rhs=(qt.T@np.where(keep,data/self.scale,0)@qf).ravel()[self.active]
        def matvec(c):
            a=self.expand(c)
            return (qt.T@(keep*(qt@a@qf.T))@qf).ravel()[self.active]+ridge*c
        diagonal=((qt**2).T@keep.astype(float)@(qf**2)).ravel()[self.active]+ridge
        op=LinearOperator((self.ncoeff,self.ncoeff),matvec=matvec,dtype=float)
        pre=LinearOperator(op.shape,matvec=lambda c:c/diagonal,dtype=float)
        trace=[]
        coeff,info=cg(op,rhs,M=pre,rtol=self.config.cg_rtol,maxiter=1000,callback=lambda _:trace.append(1))
        self.history.append(dict(info=int(info),iterations=len(trace),seconds=time.perf_counter()-start))
        if info: raise RuntimeError(f'Mixed solver CG failure: {info}')
        return (qt@self.expand(coeff)@qf.T)*self.scale


def mixed_support(solver,keep):
    start=time.perf_counter();qt,qf=solver.qt,solver.qf;nt,nf=solver.shape
    tt=np.einsum('ta,tb->abt',qt,qt).reshape(nt*nt,-1)
    moments=(tt@keep.astype(float)).reshape(nt,nt,-1)
    H=np.einsum('abf,fi,fj->aibj',moments,qf,qf,optimize=True).reshape(nt*nf,nt*nf)
    ix=np.ix_(solver.active,solver.active)
    H=H[ix];ev,v=eigh((H+H.T)*.5);ev=np.maximum(ev,0);ridge=solver.config.ridge
    inv=(v/(ev+ridge))@v.T;inv2=(v/(ev+ridge)**2)@v.T
    def diag(cov):
        full=np.zeros((nt*nf,nt*nf));full[ix]=cov
        fc=np.einsum('aibj,fi,fj->abf',full.reshape(nt,nf,nt,nf),qf,qf,optimize=True)
        return tt.T@fc.reshape(nt*nt,-1)
    var=np.maximum(diag(inv),1e-30)
    nominal=(qt**2)@solver.active_mask.astype(float)@(qf**2).T
    return np.sqrt(var/nominal),np.clip(ridge*diag(inv2)/var,0,1),time.perf_counter()-start


In [ ]:
def experiment(data, ground, cross, times, freqs, dt, sky, config, width=0., training=None):
    started = time.perf_counter()
    df_hz = float(np.median(np.diff(freqs))) * 1e6
    normalization = np.sqrt(2 * dt[:, None] * df_hz)
    domain = (freqs >= config.tested_band_mhz[0]) & (
        freqs < config.tested_band_mhz[1]
    )
    valid_input = (
        np.isfinite(data)
        & np.isfinite(ground)
        & np.isfinite(cross)
        & (data > 0)
        & (ground > 0)
        & domain[None, :]
    )
    valid = valid_input & sky[:, None]
    cross_z = cross / np.sqrt(np.maximum(data, 1) * np.maximum(ground, 1))
    cross_z *= normalization
    channels = np.rint(freqs / (df_hz / 1e6)).astype(int)
    training = sky.copy() if training is None else sky & training
    solver = MixedFit(times, freqs, data[training], config, width)
    reference = valid & training[:, None]
    model, robust_keep, robust_trace = base._robust_model(
        data, reference, solver, normalization, config
    )
    cross_model, cross_score, cross_stats = base._cross_background(
        cross_z, solver.cross_qt, reference, config
    )
    keep = robust_keep
    refit_trace = []
    for iteration in range(config.refit_rounds):
        reasons, z, _ = base._detect(
            data,
            model,
            cross_score,
            valid,
            normalization,
            freqs,
            channels,
            reference,
            config,
        )
        reject = (
            reasons["positive_auto_excess"]
            | reasons["negative_or_model_failure"]
            | reasons["band_group_trigger"]
            | reasons["comb_group_trigger"]
            | (reasons["cross_change"] & (z > 2))
        )
        updated = robust_keep & ~reject
        change = float(np.mean(updated != keep))
        keep = updated
        model = solver.solve(data, keep)
        refit_trace.append(
            {
                "round": iteration,
                "changed_fraction": change,
                "kept_fraction": float(keep.mean()),
            }
        )
        if change < config.mask_change_tol:
            break
    inflation, prior, support_seconds = mixed_support(solver, keep)
    supported = (
        (inflation <= config.support_inflation)
        & (prior <= config.prior_fraction)
        & (model > 0)
        & np.isfinite(model)
        & domain[None, :]
    )
    reasons, z, event_stats = base._detect(
        data,
        model,
        cross_score,
        valid & supported,
        normalization,
        freqs,
        channels,
        reference & supported,
        config,
    )
    reasons["non_sky_switch_state"] = np.broadcast_to(
        ~sky[:, None], data.shape
    ).copy()
    reasons["invalid_input_or_domain"] = ~valid_input
    reasons["unsupported_background"] = ~supported
    reasons["cross_change"] = (
        (cross_score > config.cross_cut) & valid_input & sky[:, None]
    )
    before_guard = np.logical_or.reduce(list(reasons.values()))
    expanded = maximum_filter(
        before_guard,
        size=(2 * config.time_guard + 1, 2 * config.frequency_guard + 1),
        mode="constant",
    )
    reasons["guard"] = expanded & ~before_guard
    flags = base.encode_reasons(reasons)
    return {
        "flags": flags,
        "model": np.where(supported, model, np.nan),
        "model_raw": model,
        "residual_z": np.where(supported, z, np.nan),
        "support_ok": supported,
        "fit_keep": keep,
        "reasons": reasons,
        "inflation": inflation,
        "prior_fraction": prior,
        "cross_background": cross_model,
        "cross_score": cross_score,
        "diagnostics": {
            "seconds": time.perf_counter() - started,
            "solver": solver.history,
            "robust_trace": robust_trace,
            "refit_trace": refit_trace,
            "cross": cross_stats,
            "support_seconds": support_seconds,
            **event_stats,
        },
    }


## Independent dense solver and support check

In [ ]:
rng=np.random.default_rng(912)
s=MixedFit.__new__(MixedFit);s.config=P;s.history=[]
s.qt=np.linalg.qr(rng.normal(size=(21,4)))[0];s.qf=np.linalg.qr(rng.normal(size=(24,6)))[0]
s.shape=(4,6);s.scale=np.ones(24);s.active_mask=np.zeros(s.shape,bool)
s.active_mask[:,:3]=True;s.active_mask[0,3:]=True
s.active=np.flatnonzero(s.active_mask.ravel());s.ncoeff=len(s.active)
y=rng.normal(size=(21,24));keep=rng.random(y.shape)>.3
pred=s.solve(y,keep)
B=np.einsum('ta,fb->tfab',s.qt,s.qf).reshape(y.size,-1)[:,s.active]
H=B[keep.ravel()].T@B[keep.ravel()]
inv=np.linalg.inv(H+P.ridge*np.eye(s.ncoeff))
expected=(B@(inv@(B[keep.ravel()].T@y[keep]))).reshape(y.shape)
np.testing.assert_allclose(pred,expected,atol=1e-7)
infl,prior,_=mixed_support(s,keep)
np.testing.assert_allclose(infl,np.sqrt(np.einsum('ni,ij,nj->n',B,inv,B)/np.sum(B**2,axis=1)).reshape(y.shape),rtol=1e-8)
print('Dense fit and joint-support checks passed.')


## Real windows, inputs and provenance

In [ ]:
idx=MetadataIndex(CAMPAIGN/'data');sel=idx.select()
meta=sel.meta.assign(visit=sel.visits(gap_s=600));t0=float(meta.loc[meta.visit==13,'time_best'].min())
cases={};provenance={}
for offset in [1,4,10]:
    selection=idx.select(time=(t0+offset*3600,t0+offset*3600+1200))
    bundles=base.load_selection_inputs(selection);a=bundles['air']
    assert a.meta.sync_consistent.all()
    assert a.meta.integration_time.nunique()==1
    assert np.allclose(np.diff(a.t),a.meta.integration_time.iloc[0],rtol=1e-5)
    cases[offset]=dict(data=np.asarray(a.data,float),ground=np.asarray(bundles['ground'].data,float),cross=bundles['cross'].data,
        times=a.t,freqs=a.freqs_mhz,dt=a.meta.integration_time.to_numpy(float),
        sky=a.meta.rfswitch.to_numpy(object)=='RFANT',config=P)
    provenance[offset]=dict(start_utc=str(pd.to_datetime(a.t[0],unit='s',utc=True)),
        end_utc=str(pd.to_datetime(a.t[-1],unit='s',utc=True)),shape=list(a.data.shape),
        phases=a.meta.filter_phase.unique().tolist(),
        keys={name:b.meta[['file','input_key','conjugated']].drop_duplicates().to_dict('records') for name,b in bundles.items()},
        input_sha256={fn:hashlib.sha256((CAMPAIGN/'data'/fn).read_bytes()).hexdigest() for fn in selection.files})
    print(offset,provenance[offset]['start_utc'],provenance[offset]['shape'])
display(provenance)


In [ ]:
runs={};records=[]
widths=[0.,150e-9,300e-9]
for offset,a in cases.items():
    for width in widths:
        label='baseline' if width==0 else f'constant {width*1e9:.0f} ns'
        print('Running',offset,label,flush=True)
        result=experiment(**a,width=width);runs[offset,label]=result
        for lo,hi in [(35,50),(50,70),(70,88),(88,108),(108,235)]:
            j=(a['freqs']>=lo)&(a['freqs']<hi)
            records.append(dict(offset_h=offset,model=label,band=f'{lo}-{hi}',
                retained=(~result['flags'].astype(bool)[:,j]).mean(),support=result['support_ok'][:,j].mean(),
                positive=result['reasons']['positive_auto_excess'][:,j].mean(),
                cross=result['reasons']['cross_change'][:,j].mean(),
                median_z=np.nanmedian(result['residual_z'][:,j]),seconds=result['diagnostics']['seconds'],
                robust_converged=result['diagnostics']['robust_trace'][-1]['changed_fraction']<P.mask_change_tol))
metrics=pd.DataFrame(records)
display(metrics[metrics.band=='50-70'])
display(metrics)


## Held-out prediction: same candidate samples for every model

The evaluation subset uses training-only medians/MADs to exclude grossly variable samples, and the baseline run’s comb classification to remove triggered teeth. This is a conservative comparison subset, **not a clean-sky truth label**. Stable RFI may remain in it. Models never train on the withheld rows.

In [ ]:
hold_records=[]
for offset in [4,10]:
    a=cases[offset];nt=len(a['times']);hold=(np.arange(nt)%37==0)|((a['times']-a['times'][0]>590)&(a['times']-a['times'][0]<610))
    med=np.median(a['data'][~hold],axis=0)
    mad=1.4826*np.median(abs(a['data'][~hold]-med),axis=0)
    rt=np.sqrt(2*a['dt'][:,None]*np.diff(a['freqs']).mean()*1e6)
    core=abs(a['data']-med)<=4*np.maximum(mad,med/rt)
    candidates=hold[:,None]&core&~runs[offset,'baseline']['reasons']['comb_group_trigger']
    for width in widths:
        label='baseline' if width==0 else f'constant {width*1e9:.0f} ns'
        print('Holdout',offset,label,flush=True)
        result=experiment(**a,width=width,training=~hold)
        for lo,hi in [(50,70),(108,235)]:
            j=(a['freqs']>=lo)&(a['freqs']<hi)
            select=candidates&j[None,:]&result['support_ok']
            z=result['residual_z'][select]
            hold_records.append(dict(offset_h=offset,model=label,band=f'{lo}-{hi}',samples=len(z),
                median_z=np.nanmedian(z),median_abs_z=np.nanmedian(abs(z)),p90_abs_z=np.nanquantile(abs(z),.9),
                supported_fraction=select.sum()/max((candidates&j[None,:]).sum(),1)))
hold_table=pd.DataFrame(hold_records);display(hold_table)


## Visual comparison, 10-hour window

Waterfall scales: data/model use counts with LogNorm 10^4–10^7; excluded is black; retained z uses a red/blue scale from −8 to +8. The fixed spectral plot is a scientific review artifact; v2.1 retains its widget-compatible interface.

In [ ]:
a=cases[10];f=a['freqs'];extent=[f[0],f[-1],20,0]
fig,axes=plt.subplots(3,4,figsize=(17,11),sharex=True,sharey=True,constrained_layout=True)
for row,label in enumerate(['baseline','constant 150 ns','constant 300 ns']):
    r=runs[10,label]
    for col,(image,title) in enumerate([(a['data'],'data'),(r['model'],'model'),(r['flags']!=0,'excluded'),(np.where(r['flags']==0,r['residual_z'],np.nan),'retained z')]):
        opts=dict(norm=LogNorm(1e4,1e7)) if col<2 else dict(cmap='gray_r',vmin=0,vmax=1) if col==2 else dict(cmap='RdBu_r',vmin=-8,vmax=8)
        axes[row,col].imshow(image,extent=extent,aspect='auto',interpolation='nearest',**opts)
        axes[row,col].set_title(label+' / '+title)
        #axes[row,col].set_xlim(40,85)
for ax in axes[-1]:ax.set_xlabel('MHz')
for ax in axes[:,0]:ax.set_ylabel('Minutes')
plt.show()
fig,axes=plt.subplots(2,1,figsize=(12,7),sharex=True,constrained_layout=True)
for label in ['baseline','constant 150 ns','constant 300 ns']:
    r=runs[10,label]
    axes[0].plot(f,np.nanmedian(r['residual_z'],axis=0),label=label)
    axes[1].plot(f,(r['flags']==0).mean(axis=0),label=label)
axes[0].set_ylim(-20,80);axes[0].set_ylabel('Median z');axes[0].axhline(6,color='k',ls=':')
axes[1].set_ylim(0,1);axes[1].set_ylabel('Retained fraction');axes[1].set_xlabel('MHz')
for ax in axes:ax.set_xlim(40,85);ax.legend()
plt.show()


## Controlled interference check

This synthetic experiment tests the specific ambiguity: a broad, stable 5% spectral feature plus persistent narrow lines, a DTV-like burst, comb events, and a cross-only event. Its known background is a test fixture, not evidence that a corresponding feature in the real observations is sky. Events are evaluated at their first integration and over their whole duration using RFI evidence bits, without support vetoes or guards counting as detections.

In [ ]:
rng=np.random.default_rng(20260919)
f=cases[10]['freqs'];nt=256;t=np.linspace(0,1200,nt);dt=np.full(nt,cases[10]['dt'][0])
rt=np.sqrt(2*dt[:,None]*np.diff(f).mean()*1e6)
truth=1e6*(1+.12*np.cos(2*np.pi*f[None,:]/150))*(1+.02*np.cos(2*np.pi*t[:,None]/1800))
truth*=1+.05*np.exp(-.5*((f[None,:]-60)/4)**2)
clean=truth*(1+rng.normal(size=truth.shape)/rt);data=clean.copy();ground=truth*.8
cz=(rng.normal(size=truth.shape)+1j*rng.normal(size=truth.shape))/np.sqrt(2)
cz+=50*np.exp(1j*2*np.pi*f[None,:]/9)
ch=np.rint(f/np.diff(f).mean()).astype(int);domain=(f>=35)&(f<235)
injected=np.zeros_like(data,bool);events=[]
def inject(name,start,stop,j,sigma,cross_sigma=0):
    rows=np.arange(start,stop);cols=np.flatnonzero(j)
    data[np.ix_(rows,cols)]+=sigma*truth[np.ix_(rows,cols)]/rt[start:stop]
    cz[np.ix_(rows,cols)]+=cross_sigma
    injected[np.ix_(rows,cols)]=True
    events.append((name,start,stop,cols))
inject('persistent narrow 61 MHz',0,nt,ch==ch[np.argmin(abs(f-61))],50)
inject('persistent FM lines',0,nt,(f>=88)&(f<108)&(ch%3==0),50)
inject('DTV-like 54-60 MHz burst',70,82,(f>=54)&(f<60),4,12)
inject('comb8',110,124,domain&(ch%8==3),4)
inject('comb4',155,169,domain&(ch%4==1),4)
inject('comb2',195,209,domain&(ch%2==0),4)
inject('cross-only 65 MHz',230,236,ch==ch[np.argmin(abs(f-65))],0,20)
cross=cz*np.sqrt(data*ground)/rt
synthetic=dict(data=data,ground=ground,cross=cross,times=t,freqs=f,dt=dt,sky=np.ones(nt,bool),config=P)
injection_records=[];clean_records=[]
for width in widths:
    label='baseline' if width==0 else f'constant {width*1e9:.0f} ns'
    print('Interference check',label,flush=True)
    r=experiment(**synthetic,width=width)
    evidence=np.logical_or.reduce([r['reasons'][k] for k in ['positive_auto_excess','cross_change','band_group_trigger','comb_group_trigger']])
    for name,start,stop,j in events:
        injection_records.append(dict(model=label,event=name,first_recall=evidence[start,j].mean(),recall=evidence[start:stop][:,j].mean()))
    valid=domain[None,:]&~injected
    low=valid&((f>=50)&(f<70))[None,:]
    err=(r['model_raw']/truth-1)*rt
    clean_records.append(dict(model=label,clean_low_retained=(r['flags'][low]==0).mean(),
        clean_low_support=r['support_ok'][low].mean(),clean_low_bias_z=np.median(err[low]),
        clean_low_rmse_z=np.sqrt(np.mean(err[low]**2)),clean_collateral=(r['flags'][valid]!=0).mean()))
injection_table=pd.DataFrame(injection_records);clean_table=pd.DataFrame(clean_records)
display(injection_table);display(clean_table)


## Results and interpretation

The constant 300 ns correction substantially improves the reported 10-hour window, but does not establish a universally safe replacement.

| 10-hour window, 50–70 MHz | Baseline | Constant 150 ns | Constant 300 ns |
| --- | ---: | ---: | ---: |
| Retained samples | 12.88% | 26.80% | 33.69% |
| Supported model | 100% | 100% | 100% |
| Median signed residual, all supported cells | 22.97 | 8.25 | 3.63 |
| Held-out median absolute residual | 27.51 | 13.71 | 6.91 |
| Experimental fit runtime, seconds | 19.9 | 57.0 | 32.4 |

Residuals are in the existing radiometer normalization. The 10-hour held-out comparison uses exactly 6,112 supported evaluation samples for each model. Its persistent bright-line tail remains large (90th-percentile absolute residual about 13,400); the evaluation pool is not a clean-sky label. Visual comparison shows substantial recovery between the bright narrow features, while those features remain excluded.

The other windows improve less: 50–70 MHz retention goes from 19.59% to 21.89% at 1 hour, and from 6.51% to 11.12% at 4 hours. The 4-hour held-out median absolute residual remains about 64 noise units for the candidate, so that window is not resolved by this change. Its supported evaluation samples also differ between models; do not interpret the small improvement there as a matched-sample validation.

**Stability tradeoff remains at the edges.** With the 300 ns correction, 35–50 MHz support drops from 100% to 93.79% at 4 hours and to 98.36% at 10 hours. Support at 70–88 MHz drops to 96.02% in the 4-hour window. Above 108 MHz, retained fractions change by less than 0.2 percentage points in these tests. FM changes indirectly through the shared fit; its recovery was not a target of this experiment.

The independent dense fit and covariance-diagonal checks passed. All nine full real-data fits reached the robust mask tolerance. All seven injected event types have 100% first-integration recall for both candidates. Whole-event DTV-like burst recall is 97.22%, versus 97.92% for the baseline; all other injected events have 100% recall. The candidate preserves 98.95% of synthetic clean low-band samples and reduces known-background low-band RMSE from 0.90 to 0.045 noise units. This one controlled realization tests specific failure modes, not general real-data cleanliness.

The experiment uses a diagonal CG preconditioner for both baseline and candidate, with the same objective and thresholds as the package; the baseline's near-threshold flags can differ by a few cells from the package's separable preconditioner. Timings are experimental, not a benchmark of an optimized implementation.

## Deferred findings

Persistent broad interference can be indistinguishable from stationary continuum in this model. More stringent real-data validation would be needed before interpreting all recovered pixels as clean sky. Edge-support losses suggest reviewing the correction's taper and regularization before adopting it globally; neither was tuned here.

## Decision requested

The 300 ns constant-time correction is a useful candidate for the 10-hour 50–70 MHz problem, with a measured stability cost elsewhere. Review whether to proceed with refining edge support and validating recovered channels before integrating an opt-in correction into v3-beta. The package defaults and v2.1 notebook have not been modified by this experiment, and no campaign products were written.

STOPPED AT REVIEW GATE — awaiting Aaron's approval.
